# 07. 다른 방식으로 계산해도 같은 결론인가 — 분석 선택에 대한 민감도
> 계획서: `계획/P7_표본_강건성_비교.md` (Codex 5차 반영 + 발표용 단순화) · 코드: `code/robust_cnt.py`

**한 줄 요약**: P5에서 *"건수가 35% 넘게 줄어든 기업은 이후 6개월 입금이 크게 줄 가능성이 같은 조건의 기업보다 약 4.7배"* 였습니다. 이 노트북은 **계산 방식을 하나씩 바꿔도** 이 배수가 어느 범위에서 움직이는지, 연관이 사라지는 조건이 있는지 봅니다.

**하는 일**: 변형마다 **한 가지만** 바꿔 P5와 같은 분석을 다시 합니다 (여러 개를 동시에 바꾸지 않습니다).
| 묶음 | 바꾸는 것 |
|---|---|
| 구간 가정 | 건수 구간을 건수로 바꾸는 가정 (50건초과를 51·60·100건으로, 구간 하한·상한, 순서 코드 합) |
| 활동량 하한 | 너무 적게 거래하는 기업을 빼는 기준 |
| 입금 약화 정의 | 하위 5% → 10%·2.5%, 추적 기간 6 → 3개월 |
| n·k | 감소 기준 25%·45%, 비교 구간 길이 k=3, 그리고 **최종 n(35%)으로 비교 기준·창 길이 선택을 처음부터 다시** |
| 비교군·층 | 비교 기업 구성, 층 나누는 방식 |
| 표본·기간 | 35개월 법인만 / 35개월 이상 / 2024·2025 기준월 나눠서 |

**읽는 법**: 변형의 95% 범위가 **1보다 크면** 그 방식에서도 연관이 유지된다는 뜻입니다. 모든 변형을 숨기지 않고 한 표·한 그림에 보입니다. *'기준 대비 비'* 는 변형 RR ÷ 기준 RR 입니다 (1에 가까우면 같은 크기). 변형 결과를 보고 기준 규칙을 바꾸지 않습니다. 화면 출력은 집계값뿐 · 5 미만 칸은 `<5` · 법인ID 없음. 몇 분 걸립니다 (`CNT_BOOT` 로 줄일 수 있음).

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "건수분석" / "code", base / "yangdaecheon" / "건수분석" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 yangdaecheon/건수분석/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
pops = {"P36": masks["P36"], "P35+": masks["P35p"]}
print({k: int(v.sum()) for k, v in pops.items()}, "개 기업")

import math
import detect_cnt as dc
import validate_cnt as vc
import robust_cnt as rb

spec = json.load(open(WORK / "interim" / "detect_spec_v2.json", encoding="utf-8"))
rule, V1 = spec["rule"], spec["v1_rule"]
MODE, K, L, PHI2, N2 = rule["비교 기준"], int(rule["k"]), int(rule["L"]), float(rule["phi"]), float(rule["n_pct"])
B = int(os.environ.get("CNT_BOOT", "2000"))
SEED = 20261301           # 같은 표본의 변형은 같은 시드(= 같은 기업 재표집)를 써서 기준과 짝지어 비교한다

def sup(x):
    return cl.sup(int(round(x)))

def f_rr(rr, lo, hi):
    return "-" if not np.isfinite(rr) else (f"{rr:.2f} ({lo:.2f}~{hi:.2f})" if np.isfinite(lo) else f"{rr:.2f}")

mk36, mk35, mk35p = masks["P36"], masks["P35"], masks["P35p"]
C5 = vc.fit_constants(panel, mk36, MODE, K, PHI2, N2)
D36 = vc.prepare(panel, mk36, C5)
assert int((D36["det"] >= 0).sum()) == spec["power"]["P36"]["N1"], "04의 탐지 기업 수와 다릅니다 (04를 다시 실행하세요)"
F36, ER = D36["F"], D36["er"]
BASE = rb.run_variant(D36, F36, B, SEED)
BASE_OBJ = {"R": BASE["R"], "D": D36}
print(f"기준(P36, v2): 탐지 {BASE['n_det']:,}곳, RR {f_rr(BASE['R']['rr'], BASE['R']['lo'], BASE['R']['hi'])}")

## 1. 비교 가능한 구간인가 — 기준월 기간별 분석 가능성
2024년 기준월과 2025년 기준월로 나눠 볼 때, 각 기간에 판정 가능한 기업-월과 탐지 기업이 충분한지, 입금 결과를 끝까지 추적할 수 있는지(추적 완료율) 먼저 확인합니다.

In [ ]:
U0 = D36["units"]
rows = []
for nm, es in (("2024년 기준월", [e for e in ER if e <= 23]), ("2025년 기준월", [e for e in ER if e >= 24]), ("전체", ER)):
    jf_cells = int((D36["status"][:, es] == 0).sum())
    det_firms = int(((D36["det"] >= 0) & np.isin(D36["det"], es)).sum())
    det_def = int((U0.is_det & np.isin(U0.e, es)).sum())
    rows.append({"기간": nm, "기준월 수": len(es), "판정 가능 기업-월": sup(jf_cells), "탐지 기업": sup(det_firms), "입금 변화율이 정의된 탐지 기업": sup(det_def),
                 "추적 완료율 %": round(100 * det_def / max(det_firms, 1), 1), "비교 기업-월": sup(int((~U0.is_det & np.isin(U0.e, es)).sum()))})
cl.show_table(pd.DataFrame(rows), "기간별 분석 가능성 (P36)", OUT / "nb07", "period_availability")

## 2. 임시 30%에서 최종 n(35%)으로 — 선택 절차를 처음부터 다시
비교 기준(직전 기간/전년 같은 달)과 창 길이 k는 처음에 **임시 감소 기준 30%** 로 골랐고, 최종 n은 그 뒤에 35%로 정해졌습니다. **최종 n(35%)으로 같은 선택 절차를 처음부터 다시** 돌려 결정이 달라지는지 봅니다 (건수만 사용, 입금은 쓰지 않음). 두 문턱에서 탐지되는 기업과 탐지월이 얼마나 같은지도 봅니다.

In [ ]:
s30, s35 = rb.redo_selection(panel, mk36, 0.30), rb.redo_selection(panel, mk36, 0.35)
rows = [{"임시 감소 기준": f"{int(100 * s['thr'])}%", "비교 기준": s["mode"], "창 길이 k": s["k"], "활동량 하한 Φ": round(s["phi"], 2), "n (%)": s["n"], "k 선택 이유": s["why"]} for s in (s30, s35)]
cl.show_table(pd.DataFrame(rows), "② 비교 기준·③ 창 길이 선택을 임시 기준 30% 와 35% 로 각각 처음부터", OUT / "nb07", "selection_redo")
rows = [{"임시 기준": f"{int(100 * s['thr'])}%", "창 길이 k": sh["k"], "탐지 기업": sup(sh["detected"]), "사후 회복(일시적 하락) 비율 %": round(100 * sh["share"], 1) if np.isfinite(sh["share"]) else None}
        for s in (s30, s35) for sh in s["shares"]]
cl.show_table(pd.DataFrame(rows), "창 길이별 일시적 하락 비율 (20% 미만인 가장 짧은 창을 고름)", OUT / "nb07", "selection_shares")
ch = D36["change"]
ov = rb.detection_overlap(dc.first_detect(ch, PHI2, 0.30), dc.first_detect(ch, PHI2, 0.35))
cl.show_table(pd.DataFrame([{"비교": f"감소 기준 30% vs 35% (같은 Φ={PHI2:.2f})", "30% 탐지 기업": sup(ov["n_a"]), "35% 탐지 기업": sup(ov["n_b"]), "둘 다 탐지": sup(ov["n_both"]), "일치도(Jaccard)": round(ov["jaccard"], 2),
                             "같은 기업 탐지월 차이 중앙값(개월)": ov["month_diff_median"], "탐지월이 같은 비율 %": round(100 * ov["same_month_share"], 1)}]), "30% 와 35% 문턱의 최초 탐지 일치도", OUT / "nb07", "threshold_overlap")
same_rule = (s35["mode"] == MODE and s35["k"] == K)
print("최종 n(35%)으로 다시 골라도 비교 기준·창 길이가 같음 → 결정 불변" if same_rule else f"※ 최종 n(35%)에서는 비교 기준 {s35['mode']}, k={s35['k']} 가 선택됨 → 아래 변형(F4)으로 같은 분석을 처음부터 다시 합니다")

## 3. 변형 실행 — 한 가지씩 바꿔서
같은 표본의 변형은 **같은 기업 재표집**으로 기준과 짝지어 비교합니다 (기준 대비 비의 범위).

In [ ]:
t0 = time.time()
RESULTS = []

def add(group, label, res, note=""):
    RESULTS.append({"group": group, "label": label, "res": res, "note": note})

def prep(**kw):
    return vc.prepare(panel, mk36, C5, **kw)

# 구간 가정 (A: 같은 문턱 Φ·n 유지)
add("구간 가정", "50건초과 = 51건 (cap 51)", rb.run_variant(prep(cap=51.0), F36, B, SEED, BASE_OBJ))
add("구간 가정", "50건초과 = 100건 (cap 100)", rb.run_variant(prep(cap=100.0), F36, B, SEED, BASE_OBJ))
add("구간 가정", "구간 하한으로 합산 (I1_low)", rb.run_variant(prep(I=rb.i1_bound(panel, mk36, "low")), F36, B, SEED, BASE_OBJ))
add("구간 가정", "구간 상한으로 합산 (I1_high, 50건초과 = 100 시나리오)", rb.run_variant(prep(I=rb.i1_bound(panel, mk36, "high")), F36, B, SEED, BASE_OBJ))
# B: 지표 단위가 달라 문턱을 다시 구함 -- 주 규칙처럼 해석하지 않는다
I2 = rb.i2_matrix(panel, mk36)
FL36 = dc.ceiling_window_flag(panel, mk36, K, dc.lag_for(MODE, K))
d2 = dc.decide_fixed(I2, MODE, K, FL36)
C2 = vc.fit_constants(panel, mk36, MODE, K, d2["phi"], d2["n"], I=I2)
D2 = vc.prepare(panel, mk36, C2, phi=d2["phi"], n_pct=d2["n"], I=I2)
add("구간 가정", f"순서 코드 합 I2 (문턱 다시 구함: Φ={d2['phi']:.1f}, n={d2['n']:.0f}%)", rb.run_variant(D2, F36, B, SEED, BASE_OBJ), "지표 단위가 달라 문턱을 다시 구한 별도 규칙 — 방향 확인용")
# 활동량 하한
add("활동량 하한", f"원래 등록 규칙 v1 (Φ={V1['phi']:.1f}, n={V1['n_pct']:.0f}%)", rb.run_variant(prep(phi=float(V1["phi"]), n_pct=float(V1["n_pct"])), F36, B, SEED, BASE_OBJ))
add("활동량 하한", "하한 거의 없음 (Φ=1)", rb.run_variant(prep(phi=1.0), F36, B, SEED, BASE_OBJ))
# 입금 약화 정의 (문턱은 P36 판정 가능 전체에서 같은 방식으로 다시 구함)
add("입금 약화 정의", "하위 10% 로 넓힘", rb.run_variant(prep(theta_q=10.0), F36, B, SEED, BASE_OBJ))
add("입금 약화 정의", "하위 2.5% 로 좁힘", rb.run_variant(prep(theta_q=2.5), F36, B, SEED, BASE_OBJ))
add("입금 약화 정의", "추적 기간 3개월", rb.run_variant(prep(post=(1, 3)), F36, B, SEED, BASE_OBJ))
# n·k
add("n·k", "감소 기준 n = 25%", rb.run_variant(prep(n_pct=25.0), F36, B, SEED, BASE_OBJ))
add("n·k", "감소 기준 n = 45%", rb.run_variant(prep(n_pct=45.0), F36, B, SEED, BASE_OBJ))
d3 = dc.decide_fixed(dc.i1_matrix(panel, mk36), MODE, 3, dc.ceiling_window_flag(panel, mk36, 3, dc.lag_for(MODE, 3)))
C3 = vc.fit_constants(panel, mk36, MODE, 3, d3["phi"], d3["n"])
add("n·k", f"비교 구간 k = 3 (Φ={d3['phi']:.1f}, n={d3['n']:.0f}% 다시 구함)", rb.run_variant(vc.prepare(panel, mk36, C3), F36, B, SEED, BASE_OBJ), "k 를 바꾸면 Φ·n 도 그 k 에서 다시 구한 규칙")
if not same_rule:
    Cn = vc.fit_constants(panel, mk36, s35["mode"], s35["k"], s35["phi"], s35["n"])
    add("n·k", f"최종 n(35%)으로 다시 고른 규칙 ({s35['mode']}, k={s35['k']})", rb.run_variant(vc.prepare(panel, mk36, Cn), F36, B, SEED, BASE_OBJ))
# 비교군·층
add("비교군·층", "보조 비교군: 한 번도 탐지되지 않은 기업 (RR 이 부풀 수 있음)", rb.run_variant(D36, F36, B, SEED, BASE_OBJ, units=rb.never_detected_controls(D36["units"], D36["det"])))
add("비교군·층", "층: 기준월을 합쳐 규모 × 업종군", rb.run_variant(D36, F36, B, SEED, BASE_OBJ, cells=rb.cells_pooled_months(D36["units"])))
add("비교군·층", "층: 업종군 없이 기준월 × 규모", rb.run_variant(D36, F36, B, SEED, BASE_OBJ, cells=rb.cells_no_sector(D36["units"])))
# 표본·기간
Dh = vc.prepare(panel, mk35, C5)
add("표본·기간", "35개월 법인만 (P35만, 독립 기업)", rb.run_variant(Dh, Dh["F"], B, 20261302))
Dp = vc.prepare(panel, mk35p, C5)
add("표본·기간", "35개월 이상 전체 (P35+, 빈 달은 채우지 않음)", rb.run_variant(Dp, Dp["F"], B, 20261303), "P36 을 포함하므로 독립 증거가 아님")
add("표본·기간", "2024년 기준월만", rb.run_variant(D36, F36, B, SEED, units=U0.sub(U0.e <= 23)))
add("표본·기간", "2025년 기준월만", rb.run_variant(D36, F36, B, SEED, units=U0.sub(U0.e >= 24)))
print(f"변형 {len(RESULTS)}개 계산 완료 ({time.time() - t0:.0f}초)")

## 4. 한눈에 보기

In [ ]:
rows = [{"묶음": "기준", "변형": "기준 (P36, v2)", "탐지 기업": sup(BASE["n_det"]), "탐지군 약화율 %": round(100 * BASE["R"]["p_det"], 1), "비교군 약화율 % (표준화)": round(100 * BASE["R"]["p_ctrl"], 1),
         "RR (95% 범위)": f_rr(BASE["R"]["rr"], BASE["R"]["lo"], BASE["R"]["hi"]), "읽는 법": "기준"}]
for it in RESULTS:
    res, R = it["res"], it["res"]["R"]
    r = {"묶음": it["group"], "변형": it["label"], "탐지 기업": sup(res["n_det"]), "탐지군 약화율 %": round(100 * R["p_det"], 1) if np.isfinite(R["p_det"]) else None,
         "비교군 약화율 % (표준화)": round(100 * R["p_ctrl"], 1) if np.isfinite(R["p_ctrl"]) else None, "RR (95% 범위)": f_rr(R["rr"], R["lo"], R["hi"])}
    r["기준 대비 비 (95% 범위)"] = f_rr(*res["ratio"]) if "ratio" in res else "-"
    ovl = res.get("overlap")
    r["최초 탐지 일치(Jaccard)"] = round(ovl["jaccard"], 2) if ovl else None
    r["탐지월 차이 중앙값(개월)"] = ovl["month_diff_median"] if ovl and np.isfinite(ovl["month_diff_median"]) else None
    r["읽는 법"] = ("탐지 30곳 미만 → 참고만" if res["descriptive"] else ("연관 유지 (범위 하한 > 1)" if (np.isfinite(R["lo"]) and R["lo"] > 1) else "범위가 1을 포함하거나 정의 불가 → 연관이 확인되지 않는 조건")) + (f" · {it['note']}" if it["note"] else "")
    rows.append(r)
TAB = pd.DataFrame(rows)
cl.show_table(TAB, "변형별 결과 (같은 표본의 변형은 기준과 짝지은 비교)", OUT / "nb07", "variants")
items = [("기준 (P36, v2)", BASE["R"]["rr"], BASE["R"]["lo"], BASE["R"]["hi"], "기준")] + [(it["label"], it["res"]["R"]["rr"], it["res"]["R"]["lo"], it["res"]["R"]["hi"], it["group"]) for it in RESULTS if np.isfinite(it["res"]["R"]["rr"])]
colors = {"기준": "#eb6834", "구간 가정": "#2a78d6", "활동량 하한": "#1b9e77", "입금 약화 정의": "#7570b3", "n·k": "#e6ab02", "비교군·층": "#666666", "표본·기간": "#d95f02"}
fig, ax = plt.subplots(figsize=(8.5, 0.34 * len(items) + 1.4))
for i, (lab, rr, lo, hi, g) in enumerate(items[::-1]):
    ax.plot([lo, hi], [i, i], color=colors.get(g, "#2a78d6"), lw=2.5, solid_capstyle="round")
    ax.plot([rr], [i], "o", color=colors.get(g, "#2a78d6"), ms=6)
    ax.text(hi + 0.2, i, f"{rr:.1f}", va="center", fontsize=8)
ax.axvline(1, color="k", ls="--", lw=1)
ax.axvline(BASE["R"]["rr"], color="#eb6834", ls=":", lw=1)
ax.set_yticks(range(len(items))); ax.set_yticklabels([x[0][:34] for x in items[::-1]], fontsize=8)
ax.set_xlabel("입금이 크게 줄 가능성의 배수 (점 = 추정, 선 = 95% 범위, 검은 점선 = 1배, 주황 점선 = 기준)")
ax.set_title("어떤 방식으로 계산해도 신호는 연관된다 — 변형별 배수 (P36)")
plt.tight_layout(); plt.show()
vr = [it for it in RESULTS if not it["res"]["descriptive"] and np.isfinite(it["res"]["R"]["rr"])]
lows = [it["label"] for it in vr if not (it["res"]["R"]["lo"] > 1)]
print(f"변형 {len(vr)}개의 RR 은 {min(it['res']['R']['rr'] for it in vr):.2f} ~ {max(it['res']['R']['rr'] for it in vr):.2f} 배 범위에서 움직였습니다 (기준 {BASE['R']['rr']:.2f}).")
print("95% 범위가 1을 포함하는(연관이 확인되지 않는) 변형:", lows if lows else "없음")

## 5. 요약 저장

In [ ]:
def pk(res):
    R = res["R"]
    d = {k: (None if not np.isfinite(R[k]) else float(R[k])) for k in ("rr", "lo", "hi", "p_det", "p_ctrl")}
    d["n_det"] = int(res["n_det"])
    d["ratio"] = None if "ratio" not in res else [float(x) for x in res["ratio"]]
    d["jaccard"] = None if not res.get("overlap") else float(res["overlap"]["jaccard"])
    return d

summary = {"base": pk(BASE), "variants": {f"{it['group']} | {it['label']}": pk(it["res"]) for it in RESULTS},
           "selection": {"s30": {k: v for k, v in s30.items() if k != "shares"}, "s35": {k: v for k, v in s35.items() if k != "shares"}, "same_rule": bool(same_rule)},
           "threshold_overlap": {k: (None if (isinstance(v, float) and not np.isfinite(v)) else v) for k, v in ov.items()}, "synthetic": bool(panel.synthetic)}
cl.save_json(WORK / "interim" / "nb07_summary.json", summary)
print("저장: interim/nb07_summary.json,", "output/nb07/*.csv")

## 마치면
실행 후 **저장(Ctrl+S)** 만 해 주세요. 결과 표는 `data/work_cnt/output/nb07/`에 자동 저장되어 Claude가 직접 읽고 해석합니다.